In [1]:
import numpy as np

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joshfjelstul/world-cup-database")

print("Path to dataset files:", path)

c:\projects\vm2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\havar\.cache\kagglehub\datasets\joshfjelstul\world-cup-database\versions\1


In [4]:
matches_csv = os.path.join(path, "matches.csv")
df_matches = pd.read_csv(matches_csv)

In [5]:
df_matches[['home_team', 'away_team']] = df_matches['match_name'].str.extract(r'^(.*?)\s+v\s+(.*?)$')
df_matches[['home_goal', 'away_goal']] = df_matches['score'].str.extract(r'^(\d+)[\u2013-](\d+)$').astype(int)

df_matches[['home_team', 'away_team', 'home_goal', 'away_goal']].head()

df_matches[['home_team', 'away_team']] = df_matches[['home_team', 'away_team']].replace(
    {'East Germany': 'Germany', 'West Germany': 'Germany', 'Czech Republic': 'Czechia'}
)


In [6]:
vm_kvalikk_df = pd.read_csv("vm_kvalikk.csv")
vm_kvalikk_df.head()

,Home team,Away team,Home goal,Away goal
0,Paraguay,Peru,0,0
1,Colombia,Venezuela,1,0
2,Argentina,Ecuador,1,0
3,Uruguay,Chile,3,1
4,Brazil,Bolivia,5,1


In [7]:
unike_lag_historikk = list(set(df_matches['home_team']))
unike_lag_kvalikk = list(set(vm_kvalikk_df['Home team']).union(set(vm_kvalikk_df['Away team'])))

unike_lag_historikk_samlet = set(unike_lag_historikk).union(set(unike_lag_kvalikk))


with open("wc_teams.txt", "r", encoding="utf-8") as f:
    wc_teams_2026 = [line.strip() for line in f if line.strip()]

In [8]:
import difflib
import re
import unicodedata

def normalize(name):
    text = unicodedata.normalize("NFKD", name)
    text = re.sub(r"[\u0300-\u036f]", "", text)
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

hist = unike_lag_historikk_samlet
teams_2026 = wc_teams_2026

hist_norm = {team: normalize(team) for team in hist}
teams26_norm = {team: normalize(team) for team in teams_2026}
teams26_norm_inv = {norm: team for team, norm in teams26_norm.items()}
hist_norm_inv = {norm: team for team, norm in hist_norm.items()}

def best_match(norm_name, candidates):
    best_candidate = None
    best_score = 0.0
    for cand in candidates:
        score = difflib.SequenceMatcher(None, norm_name, cand).ratio()
        if score > best_score:
            best_score = score
            best_candidate = cand
    return best_candidate, best_score

hist_not_in_2026 = []
hist_suggestions = []
for team, norm in hist_norm.items():
    if norm not in teams26_norm.values():
        best, score = best_match(norm, teams26_norm.values())
        hist_not_in_2026.append(team)
        hist_suggestions.append((team, score, teams26_norm_inv.get(best)))

teams26_not_in_hist = []
teams26_suggestions = []
for team, norm in teams26_norm.items():
    if norm not in hist_norm.values():
        best, score = best_match(norm, hist_norm.values())
        teams26_not_in_hist.append(team)
        teams26_suggestions.append((team, score, hist_norm_inv.get(best)))

print("Teams in history not in 2026:")
for team in sorted(hist_not_in_2026):
    print("-", team)

print("\nTeams in 2026 not in history:")
for team in sorted(teams26_not_in_hist):
    print("-", team)

print("\nFuzzy match suggestions for history-only teams:")
for team, score, match in sorted(hist_suggestions, key=lambda x: (-x[1], x[0])):
    print(f"- {team} -> {match} ({score:.2f})")

print("\nFuzzy match suggestions for 2026-only teams:")
for team, score, match in sorted(teams26_suggestions, key=lambda x: (-x[1], x[0])):
    print(f"- {team} -> {match} ({score:.2f})")

Teams in history not in 2026:
- Afghanistan
- Albania
- Andorra
- Angola
- Antigua & Barbuda
- Armenia
- Aruba
- Azerbaijan
- Bahrain
- Bangladesh
- Barbados
- Belarus
- Belize
- Benin
- Bermuda
- Bhutan
- Bolivia
- Bosnia & Herzegovina
- Botswana
- British Virgin Islands
- Brunei
- Bulgaria
- Burkina Faso
- Burundi
- Cambodia
- Cameroon
- Cape Verde
- Cayman Islands
- Central African Republic
- Chad
- Chile
- China
- Chinese Taipei
- Comoros
- Congo
- Costa Rica
- Cuba
- Cyprus
- Czech Republic
- Czechoslovakia
- DR Congo
- Denmark
- Djibouti
- Dominica
- Dominican Republic
- El Salvador
- Equatorial Guinea
- Estonia
- Eswatini
- Ethiopia
- Faroe Islands
- Fiji
- Finland
- Gabon
- Gambia
- Georgia
- Gibraltar
- Greece
- Grenada
- Guam
- Guatemala
- Guinea
- Guinea-Bissau
- Guyana
- Honduras
- Hong Kong
- Hungary
- Iceland
- India
- Indonesia
- Israel
- Italy
- Jamaica
- Kazakhstan
- Kenya
- Kosovo
- Kuwait
- Kyrgyzstan
- Laos
- Latvia
- Lebanon
- Lesotho
- Liberia
- Libya
- Liechtenst